Análise de Vendas - Loja de Chocolates




## 🎯 Objetivo

Analisar o desempenho de vendas ao longo do ano, identificando sazonalidade,
produtos mais lucrativos e oportunidades de negócio.

## 📌 Contexto

Este dataset simula uma empresa do setor de chocolates com faturamento anual,
incluindo efeitos sazonais como Páscoa e Natal.

## 📊 Análise Geral

In [ ]:
import pandas as pd
df = pd.read_csv('../data/vendas.csv')
df.head()

In [ ]:
desc = df.describe()

# função formato brasileiro
def formato_brl(x):
    return f'R$ {x:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

# aplicar formatação só nas colunas de dinheiro
colunas_dinheiro = ['preco', 'custo', 'faturamento', 'custo_total', 'lucro']

desc_formatado = desc.copy()

for col in colunas_dinheiro:
    desc_formatado[col] = desc[col].apply(formato_brl)

# quantidade fica normal (sem R$)
desc_formatado['quantidade'] = desc['quantidade'].round(0)

desc_formatado

## 💰 Indicadores Gerais

In [ ]:
faturamento_total = df['faturamento'].sum()
lucro_total = df['lucro'].sum()

print(f'Faturamento total: R$ {faturamento_total:,.2f}')
print(f'Lucro total: R$ {lucro_total:,.2f}')

## 📊 Faturamento por Mês

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# criar coluna de mês
df['mes'] = pd.to_datetime(df['data']).dt.month

# agrupar
faturamento_mes = df.groupby('mes')['faturamento'].sum()

# função para formato brasileiro
def formato_brl(x, pos):
    return f'R$ {x:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

# gráfico
plt.figure(figsize=(12,6))

bars = plt.bar(faturamento_mes.index, faturamento_mes.values)

# título e labels
plt.title('Faturamento por Mês', fontsize=14, pad=15)
plt.xlabel('Mês', fontsize=11)
plt.ylabel('Faturamento (R$)', fontsize=11)

# aplicar formatação brasileira no eixo Y
plt.gca().yaxis.set_major_formatter(mtick.FuncFormatter(formato_brl))

# melhorar espaçamento
plt.xticks(faturamento_mes.index)
plt.tight_layout()

for bar in bars:
    y = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        y + (max(faturamento_mes.values) * 0.02),  # espaçamento proporcional
        formato_brl(y, None),
        ha='center',
        va='bottom',
        fontsize=8,
        rotation=45  # inclina pra não sobrepor
    )

plt.show()

### 💡 Insight

Observa-se um aumento significativo no faturamento nos meses 4 e 12, indicando forte influência de datas sazonais como Páscoa e Natal.

Esses períodos representam oportunidades estratégicas para maximização de receita.

## 🏆 Top Produtos por Faturamento

In [ ]:
top_produtos = df.groupby('produto')['faturamento'].sum().sort_values(ascending=False)

# função formato brasileiro
def formato_brl(x):
    return f'R$ {x:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

# aplicar formatação
top_produtos_formatado = top_produtos.apply(formato_brl)

top_produtos_formatado

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# função formato brasileiro
def formato_brl(x, pos):
    return f'R$ {x:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

plt.figure(figsize=(12,6))

bars = plt.bar(top_produtos.index, top_produtos.values)

plt.title('Top Produtos por Faturamento', fontsize=14, pad=15)
plt.xlabel('Produto')
plt.ylabel('Faturamento (R$)')

# eixo Y formatado
plt.gca().yaxis.set_major_formatter(mtick.FuncFormatter(formato_brl))

# rotacionar nomes
plt.xticks(rotation=30, ha='right')

# espaçamento automático
plt.tight_layout()

# valores nas barras
for bar in bars:
    y = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        y + (max(top_produtos.values) * 0.02),
        formato_brl(y, None),
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.ylim(0, max(top_produtos.values) * 1.15)

plt.show()

## 💰 Margem de Lucro por Produto

In [ ]:
margem = df.groupby('produto')[['faturamento','lucro']].sum()

margem['margem_%'] = (margem['lucro'] / margem['faturamento']) * 100

# função formato brasileiro
def formato_brl(x):
    return f'R$ {x:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

# aplicar formatação
margem_formatada = margem.copy()

margem_formatada['faturamento'] = margem['faturamento'].apply(formato_brl)
margem_formatada['lucro'] = margem['lucro'].apply(formato_brl)
margem_formatada['margem_%'] = margem['margem_%'].map('{:.2f}%'.format)

# ordenar pela margem
margem_formatada.sort_values(by='margem_%', ascending=False)

### 💡 Insight

A análise de margem mostra quais produtos são mais eficientes em gerar lucro.

Produtos com alta margem percentual são estratégicos, pois trazem maior retorno mesmo com menor volume de vendas.

🥇 1. 📊 GRÁFICO DE MARGEM (%)

In [ ]:
import matplotlib.pyplot as plt

# cálculo (garante que está atualizado)
margem = df.groupby('produto')[['faturamento','lucro']].sum()
margem['margem_%'] = (margem['lucro'] / margem['faturamento']) * 100

# ordenar por margem
margem_ordenada = margem.sort_values(by='margem_%', ascending=False)

# identificar produto com maior lucro (valor absoluto)
produto_top = margem['margem_%'].idxmax()

plt.figure(figsize=(12,6))

# cores (azul claro + azul forte)
cores = []
for produto in margem_ordenada.index:
    if produto == produto_top:
        cores.append('#1f77b4')   # azul forte
    else:
        cores.append('#aec7e8')   # azul claro

bars = plt.bar(margem_ordenada.index, margem_ordenada['margem_%'], color=cores)

plt.title('Margem de Lucro por Produto (%)', fontsize=14, pad=15)
plt.ylabel('Margem (%)')
plt.xticks(rotation=30, ha='right')

plt.ylim(0, margem_ordenada['margem_%'].max() * 1.2)

# valores nas barras
for bar in bars:
    y = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        y + 0.5,
        f'{y:.2f}%',
        ha='center',
        fontsize=9
    )

plt.tight_layout()
plt.show()

🥇 2. 🎯 DESTACAR PRODUTO MAIS LUCRATIVO

In [ ]:
# produto mais lucrativo (valor absoluto)
mais_lucrativo = margem.sort_values(by='lucro', ascending=False).iloc[0]

# formatador
def formato_brl(x):
    return f'R$ {x:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

print("🏆 Produto mais lucrativo:")
print(f"Produto: {mais_lucrativo.name}")
print(f"Lucro: {formato_brl(mais_lucrativo['lucro'])}")

### 💡 Insight

O produto com maior lucro absoluto é o principal motor financeiro do negócio.

Ele representa a maior contribuição direta para o resultado da empresa, sendo estratégico para campanhas e priorização comercial.

🥇 3. 🔥 FATURAMENTO vs MARGEM 


In [ ]:
plt.figure(figsize=(10,6))

plt.scatter(margem['faturamento'], margem['margem_%'])

# formatador eixo X (R$)
import matplotlib.ticker as mtick

def formato_brl(x, pos):
    return f'R$ {x:,.0f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

plt.gca().xaxis.set_major_formatter(mtick.FuncFormatter(formato_brl))

# labels
for i in margem.index:
    plt.text(
        margem.loc[i, 'faturamento'],
        margem.loc[i, 'margem_%'],
        i,
        fontsize=8
    )

plt.title('Faturamento vs Margem (%)', fontsize=14)
plt.xlabel('Faturamento (R$)')
plt.ylabel('Margem (%)')

plt.grid(True)
plt.tight_layout()
plt.show()

## 📊 Análise de Margem

Ao analisar a margem de lucro por produto, observamos que:

- A Trufa apresenta a maior margem (~62%)
- Produtos sazonais possuem alto faturamento, mas menor eficiência

Isso indica que a empresa pode melhorar sua rentabilidade focando em produtos de maior margem.

## 📊 Score

O score combina volume de vendas e eficiência operacional.

Produtos com alto score representam o melhor equilíbrio entre faturamento e margem, sendo os mais estratégicos para o negócio.

In [ ]:
# base
analise = df.groupby('produto')[['faturamento', 'lucro']].sum()
analise['margem_%'] = (analise['lucro'] / analise['faturamento']) * 100

# score combinado
analise['score'] = analise['faturamento'] * analise['margem_%']

# ordenar
analise.sort_values(by='score', ascending=False)

## 📊 Conclusão Final

A análise evidencia que o negócio possui forte dependência de sazonalidade, com picos relevantes nos períodos de Páscoa e Natal.

Produtos sazonais, especialmente o Ovo de Páscoa, concentram grande parte do faturamento, sendo fundamentais para o desempenho anual da empresa.

Por outro lado, a análise de margem mostra que nem sempre os produtos com maior volume são os mais eficientes em geração de lucro, destacando a importância de uma estratégia equilibrada entre volume e rentabilidade.

Dessa forma, recomenda-se:

- Priorizar campanhas nos períodos sazonais para maximizar receita  
- Investir em produtos com maior margem para aumentar lucratividade  
- Diversificar o portfólio para reduzir dependência de datas específicas  

Essa abordagem permite uma gestão mais estratégica e sustentável do negócio.